# 03 Feature Baselines

This notebook trains traditional machine-learning baselines using compact engineered epoch-level features. It loads reusable feature and evaluation helpers, defines repository paths, and prepares feature tables for the fixed participant-level train/validation/test split.

Model selection is performed using the training split only, with participant-level cross-validation. The validation split is used once for interim evaluation and diagnostics. The held-out test split is exported as features for later final evaluation, but it is not used here for training, tuning, permutation importance, or conclusions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

repo_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
data_processed_dir = repo_root / "data" / "processed"
raw_dir = repo_root / "data" / "raw"
epoch_index_path = repo_root / "data" / "interim" / "epoch_index.csv"
split_assignments_path = repo_root / "data" / "interim" / "split_assignments.csv"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.baselines import (  # noqa: E402
    DEFAULT_STAGE6_OUTPUT_DIR,
    balanced_sample_weights,
    feature_group_correlation_matrix,
    feature_groups_from_columns,
    grouped_permutation_importance_frame,
    permutation_importance_frame,
    plot_confusion_matrix,
)
from src.evaluate import evaluate_predictions  # noqa: E402
from src.features import (  # noqa: E402
    FEATURE_ID_COLUMNS,
    build_feature_table,
    save_feature_tables,
)
from src.preprocessing import TARGET_SLEEP_STAGE_LABELS  # noqa: E402

stage6_output_dir = repo_root / DEFAULT_STAGE6_OUTPUT_DIR
stage6_output_dir.mkdir(parents=True, exist_ok=True)

## Build Or Load Features

This section loads existing engineered feature CSVs from `data/processed/` when available. If they are missing, it builds feature tables from `data/raw/`, `data/interim/epoch_index.csv`, and `data/interim/split_assignments.csv` using `src.features.build_feature_table`.

The section saves separate train, validation, and test feature CSVs and reports a small split summary with epoch and participant counts. The test feature table is saved only as an artifact for future final evaluation.

In [ ]:
feature_paths = {
    "train": data_processed_dir / "features_train.csv",
    "validation": data_processed_dir / "features_val.csv",
    "test": data_processed_dir / "features_test.csv",
}

if all(path.exists() for path in feature_paths.values()):
    train_df = pd.read_csv(feature_paths["train"], dtype={"participant_id": str})
    val_df = pd.read_csv(feature_paths["validation"], dtype={"participant_id": str})
    test_df = pd.read_csv(feature_paths["test"], dtype={"participant_id": str})
else:
    features = build_feature_table(
        raw_dir=raw_dir,
        epoch_index_path=epoch_index_path,
        split_assignments_path=split_assignments_path,
    )
    train_df = features[features["split"] == "train"].reset_index(drop=True)
    val_df = features[features["split"] == "validation"].reset_index(drop=True)
    test_df = features[features["split"] == "test"].reset_index(drop=True)
    save_feature_tables(train_df, val_df, test_df, data_processed_dir)

display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_epochs": [len(train_df), len(val_df), len(test_df)],
    "n_participants": [
        train_df["participant_id"].nunique(),
        val_df["participant_id"].nunique(),
        test_df["participant_id"].nunique(),
    ],
}))

## Validation Setup

This section separates identifier columns from model features, creates train and validation `X/y` objects, and displays a group-level training-feature correlation heatmap. It configures `GroupKFold` so cross-validation folds are split by `participant_id`.

The section also defines explicit macro-F1 scorers for string and encoded labels, avoiding scikit-learn’s binary-F1 defaults. The final helper records validation metrics and confusion matrices in a consistent format.

In [ ]:
feature_columns = [
    column for column in train_df.columns if column not in FEATURE_ID_COLUMNS
]
X_train = train_df[feature_columns]
y_train = train_df["label"]
groups_train = train_df["participant_id"]

X_val = val_df[feature_columns]
y_val = val_df["label"]

feature_groups = feature_groups_from_columns(feature_columns)
feature_group_correlation = feature_group_correlation_matrix(X_train, feature_groups)
feature_group_correlation.to_csv(
    stage6_output_dir / "feature_group_correlation_matrix.csv"
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    feature_group_correlation,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Mean absolute pairwise Pearson correlation"},
    ax=ax,
)
ax.set_xlabel("Compared feature group")
ax.set_ylabel("Feature group")
ax.set_title("Mean Absolute Pairwise Feature Correlation by Group")
fig.tight_layout()
fig.savefig(
    stage6_output_dir / "feature_group_correlation_matrix.png",
    bbox_inches="tight",
    dpi=150,
)
display(fig)
plt.close(fig)

n_splits = min(5, groups_train.nunique())
if n_splits < 2:
    raise ValueError(
        "Participant-level cross-validation needs at least two training participants."
    )

cv = GroupKFold(n_splits=n_splits)
# Sample weights are used to balance the XGBoost training loss only; model
# selection and validation diagnostics intentionally use unweighted macro-F1.
def labeled_macro_f1_score(y_true, y_pred, sample_weight=None):
    return f1_score(
        y_true,
        y_pred,
        average="macro",
        labels=TARGET_SLEEP_STAGE_LABELS,
        zero_division=0,
    )

def unlabeled_macro_f1_score(y_true, y_pred, sample_weight=None):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

macro_f1 = make_scorer(labeled_macro_f1_score, response_method="predict")
macro_f1_unlabeled = make_scorer(unlabeled_macro_f1_score, response_method="predict")
metrics_rows = []
confusion_matrices = {}
feature_importances = {}
group_importances = {}

def record_validation_result(model_name, predictions):
    metrics, matrix = evaluate_predictions(
        y_val,
        predictions,
        model_name=model_name,
        split="validation",
    )
    metrics_rows.append(metrics)
    confusion_matrices[model_name] = matrix
    return metrics, matrix

## Majority-Class Baseline

This sanity-check model always predicts the most frequent training label. It is fit on the training feature table and evaluated once on the validation feature table.

The expected outputs are one validation metrics row and a labeled confusion matrix. These results establish the minimum useful benchmark for later models.

In [ ]:
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train, y_train)
majority_predictions = majority_model.predict(X_val)
majority_metrics, majority_confusion = record_validation_result(
    "majority_class",
    majority_predictions,
)
display(pd.DataFrame([majority_metrics]))
display(majority_confusion)

## Elastic-Net Multinomial Logistic Regression

This section fits a scikit-learn pipeline with median imputation, standardization, and balanced multinomial logistic regression with elastic-net regularization. `GridSearchCV` tunes `C` and `l1_ratio` on the training split only, using participant-level cross-validation.

The selected pipeline is then evaluated once on the validation split. Expected outputs include the top cross-validation settings, one validation metrics row, a confusion matrix, and validation-set permutation importance at both the feature and feature-group levels.

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                solver="saga",
                class_weight="balanced",
                max_iter=10000,
                random_state=42,
            ),
        ),
    ]
)

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.0, 0.5, 1.0],
    },
    scoring=macro_f1,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
logistic_search.fit(X_train, y_train, groups=groups_train)
logistic_predictions = logistic_search.predict(X_val)
logistic_metrics, logistic_confusion = record_validation_result(
    "logistic_elasticnet",
    logistic_predictions,
)
logistic_importance = permutation_importance_frame(
    logistic_search.best_estimator_,
    X_val,
    y_val,
    scoring=macro_f1,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
logistic_group_importance = grouped_permutation_importance_frame(
    logistic_search.best_estimator_,
    X_val,
    y_val,
    feature_groups,
    scoring=macro_f1,
    n_repeats=10,
    random_state=42,
)
feature_importances["logistic_elasticnet"] = logistic_importance
group_importances["logistic_elasticnet"] = logistic_group_importance

display(pd.DataFrame(logistic_search.cv_results_).sort_values("rank_test_score").head(10))
display(pd.DataFrame([logistic_metrics]))
display(logistic_confusion)
display(logistic_importance.head(30))
display(logistic_group_importance)

## XGBoost 

This section trains an XGBoost classifier using the engineered features. Labels are encoded for XGBoost, and balanced sample weights are computed from the training labels. `GridSearchCV` tunes a modest set of tree depth, learning rate, sampling, and regularization settings on the training split only, using participant-level cross-validation.

The selected model is then evaluated once on the validation split. Expected outputs include the top cross-validation settings, one validation metrics row, a confusion matrix, and validation-set permutation importance at both the feature and feature-group levels.

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
xgb_sample_weight = balanced_sample_weights(y_train_encoded)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
)
xgb_search = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1.0, 5.0],
        "min_child_weight": [1, 5],
    },
    scoring=macro_f1_unlabeled,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(
    X_train,
    y_train_encoded,
    groups=groups_train,
    sample_weight=xgb_sample_weight,
)
xgb_predictions = label_encoder.inverse_transform(xgb_search.predict(X_val))
xgb_metrics, xgb_confusion = record_validation_result(
    "xgboost_all_features",
    xgb_predictions,
)

xgb_importance = permutation_importance_frame(
    xgb_search.best_estimator_,
    X_val,
    y_val_encoded,
    scoring=macro_f1_unlabeled,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
xgb_group_importance = grouped_permutation_importance_frame(
    xgb_search.best_estimator_,
    X_val,
    y_val_encoded,
    feature_groups,
    scoring=macro_f1_unlabeled,
    n_repeats=10,
    random_state=42,
)
feature_importances["xgboost_all_features"] = xgb_importance
group_importances["xgboost_all_features"] = xgb_group_importance

display(pd.DataFrame(xgb_search.cv_results_).sort_values("rank_test_score").head(10))
display(pd.DataFrame([xgb_metrics]))
display(xgb_confusion)
display(xgb_importance.head(30))
display(xgb_group_importance)

## Interim Validation Summary

This final section combines validation metrics from the baseline models, saves artifacts under `results/stage6_feature_baselines/`, and displays each validation confusion matrix.

It saves logistic-regression and XGBoost confusion matrices as both CSV and PNG files, and writes feature-level and feature-group validation-set permutation importance for each model. These outputs are interim validation diagnostics, not final test-set comparisons.

In [ ]:
metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df.sort_values("macro_f1", ascending=False))

metrics_output_path = stage6_output_dir / "validation_metrics.csv"
metrics_df.to_csv(metrics_output_path, index=False)
print(f"Saved validation metrics to {metrics_output_path}")

saved_confusion_models = {"logistic_elasticnet", "xgboost_all_features"}
for model_name, matrix in confusion_matrices.items():
    display(model_name)
    display(matrix)
    if model_name in saved_confusion_models:
        matrix.to_csv(
            stage6_output_dir / f"validation_confusion_matrix_{model_name}.csv"
        )
        plot_confusion_matrix(
            matrix,
            stage6_output_dir / f"validation_confusion_matrix_{model_name}.png",
        )

for model_name, importance in feature_importances.items():
    importance_output_path = (
        stage6_output_dir / f"validation_permutation_importance_{model_name}.csv"
    )
    importance.to_csv(importance_output_path, index=False)
    print(f"Saved feature-level permutation importance to {importance_output_path}")

for model_name, importance in group_importances.items():
    importance_output_path = (
        stage6_output_dir
        / f"validation_group_permutation_importance_{model_name}.csv"
    )
    importance.to_csv(importance_output_path, index=False)
    print(f"Saved group-level permutation importance to {importance_output_path}")